# all-reduce-eval-metrics — ex2: sample-count-weighted eval mean via packed all_reduce

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `all-reduce-eval-metrics`. Running the final beacon cell reports progress against the `Distributed: all_reduce eval metrics` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce eval metrics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-eval-metrics`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-eval-metrics"
DD_SUBTOPIC = "Distributed: all_reduce eval metrics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Sample-count-weighted eval mean (two all_reduces)

Ex1 averaged a SCALAR loss with `all_reduce(SUM) / world_size`. That assumes every rank evaluated the SAME number of samples. In real distributed eval, the last batch is short — rank 3 might see 13 samples while ranks 0–2 see 32 each. Naive `mean` over-weights rank 3.

Correct form: reduce `(sum_loss, count)` separately, then divide:

```python
stats = t.tensor([local_loss_sum, local_count], dtype=t.float32)
dist.all_reduce(stats, op=dist.ReduceOp.SUM)
global_mean = stats[0] / stats[1]
```

**Why one tensor, not two all_reduces.** Bandwidth — one network round-trip vs two. The two scalars get packed into a length-2 tensor and reduced together. Identical math result, half the latency.

**Trap.** `local_loss_sum` (NOT `local_mean`). If you reduce the per-rank MEAN you lose the count weight and we're back to ex1's bug. The numerator must be the unreduced sum.

### Exercise 2 — sample-count-weighted eval mean via packed all_reduce

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply a packed `(sum_loss, count)` `all_reduce(SUM)` followed by a single divide to compute the true global mean of an eval metric across ranks with unequal sample counts.
> Keywords: all_reduce, weighted-mean, uneven-batch, packed-tensor
> ```

**KCs targeted:** `pack-sum-and-count-into-one-tensor`, `weighted-mean-via-sum-divide`

Implement `ex2_weighted_eval_mean(rank, world_size, dist_module, local_loss_sum, local_count)`. The uneven-batch-correct version of ex1's eval mean:

1. Pack `(local_loss_sum, local_count)` into a single length-2 tensor: `stats = t.tensor([local_loss_sum, float(local_count)], dtype=t.float32)`. (Cast count to float — `all_reduce` requires a float tensor.)
2. Run ONE `all_reduce(SUM)` over the packed tensor. After this, `stats[0]` is the global sum of losses and `stats[1]` is the global sample count.
3. Compute the true weighted mean: `global_mean = stats[0] / stats[1]`. (Returned as a Python float via `.item()`.)
4. Return `global_mean`.

Why packed.** Two all_reduces double the network round-trip cost; one packed all_reduce halves it. Math is identical.

Why this matters (the bug ex1 hides).** If rank 0 sees 32 samples with mean loss 1.0 and rank 1 sees 8 samples with mean loss 5.0:
- Naive `mean of means` = (1 + 5) / 2 = 3.0 (WRONG — over-weights small rank).
- Weighted = (32*1 + 8*5) / (32 + 8) = 72/40 = 1.8 (RIGHT — every sample counted once).

Input: `rank`, `world_size` — ints; `dist_module` — torch.distributed or mock; `local_loss_sum` — float (sum, not mean!); `local_count` — int.
Output: `float` — true global mean, same on every rank.

In [ ]:
def ex2_weighted_eval_mean(rank: int, world_size: int, dist_module,
                           local_loss_sum: float, local_count: int) -> float:
    """Weighted global mean via packed (sum, count) all_reduce(SUM) + divide."""
    raise NotImplementedError()


def _test_ex2():

    import threading
    import contextlib
    import types as _types
    from unittest.mock import patch
    import torch as _t_for_fake
    import torch.distributed as _dist_real

    class _FakeReduceOp:
        SUM = 'SUM'
        MAX = 'MAX'
        MIN = 'MIN'
        PRODUCT = 'PROD'

    class _FakeWorld:
        """Shared state across `world_size` rank-threads."""
        def __init__(self, world_size):
            self.world_size = world_size
            self.barrier = threading.Barrier(world_size)
            self.lock = threading.Lock()
            # scratch[op_id] -> list of (rank, tensor); reset per op via barrier
            self.scratch = {}
            # per-rank thread-local pinned rank
            self.tls = threading.local()
            # collected per-rank results (for the test to read)
            self.results = [None] * world_size
        def all_reduce(self, tensor, op='SUM'):
            rank = self.tls.rank
            # phase 1: every rank deposits its tensor copy
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('ar', [None] * self.world_size)
                self.scratch['ar'][rank] = tensor.detach().clone()
            self.barrier.wait()
            # phase 2: every rank reads-out the reduced result (same math)
            bag = self.scratch['ar']
            if op == 'SUM':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced + x
            elif op == 'MAX':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.maximum(reduced, x)
            elif op == 'MIN':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.minimum(reduced, x)
            elif op == 'PROD':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced * x
            else:
                raise ValueError(f'unknown fake op {op!r}')
            # mutate in-place so caller's tensor reflects the reduction
            tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('ar', None)
            self.barrier.wait()
        def reduce(self, tensor, dst, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('rd', [None] * self.world_size)
                self.scratch['rd'][rank] = tensor.detach().clone()
            self.barrier.wait()
            # only the dst rank gets the reduced result
            if rank == dst:
                bag = self.scratch['rd']
                if op == 'SUM':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced + x
                elif op == 'MAX':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.maximum(reduced, x)
                elif op == 'MIN':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.minimum(reduced, x)
                elif op == 'PROD':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced * x
                else:
                    raise ValueError(f'unknown fake op {op!r}')
                tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('rd', None)
            self.barrier.wait()
        def broadcast(self, tensor, src):
            rank = self.tls.rank
            self.barrier.wait()
            if rank == src:
                with self.lock:
                    self.scratch['bc'] = tensor.detach().clone()
            self.barrier.wait()
            if rank != src:
                tensor.copy_(self.scratch['bc'])
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('bc', None)
            self.barrier.wait()
        def barrier_op(self):
            self.barrier.wait()

    def _run_fake_world(worker_fn, world_size, *extra_args, timeout=30):
        world = _FakeWorld(world_size)
        errors = [None] * world_size
        def _runner(rank):
            world.tls.rank = rank
            # Build the fake `dist` module facade.
            fake_dist = _types.SimpleNamespace()
            fake_dist.ReduceOp = _FakeReduceOp
            fake_dist.all_reduce = lambda tensor, op='SUM': world.all_reduce(tensor, op)
            fake_dist.reduce = lambda tensor, dst, op='SUM': world.reduce(tensor, dst, op)
            fake_dist.broadcast = lambda tensor, src: world.broadcast(tensor, src)
            fake_dist.barrier = world.barrier_op
            fake_dist.get_rank = lambda: rank
            fake_dist.get_world_size = lambda: world_size
            fake_dist.init_process_group = lambda **kw: None
            fake_dist.destroy_process_group = lambda: None
            # Inject into the worker's calling globals.
            # The student code calls `dist.<op>`; we patch the `dist` name
            # in the calling namespace via direct globals injection.
            try:
                worker_fn(rank, world_size, fake_dist, world)
            except BaseException as e:
                import traceback as _tb
                errors[rank] = (e, _tb.format_exc())
        threads = [threading.Thread(target=_runner, args=(r,), daemon=True) for r in range(world_size)]
        for th in threads:
            th.start()
        for th in threads:
            th.join(timeout=timeout)
        for r, err in enumerate(errors):
            if err is not None:
                raise RuntimeError(f'rank {r} failed: {err[0]!r}\n{err[1]}')
        return world.results


    # The classic uneven-batch case.
    # Rank 0: 32 samples, mean=1.0 (sum=32). Rank 1: 8 samples, mean=5.0 (sum=40).
    # Weighted mean = (32 + 40) / (32 + 8) = 72/40 = 1.8.
    _rank_sums   = [32.0, 40.0]
    _rank_counts = [32, 8]

    def _worker(rank, world_size, dist_module, world):
        result = ex2_weighted_eval_mean(rank, world_size, dist_module,
                                        _rank_sums[rank], _rank_counts[rank])
        world.results[rank] = result

    results = _run_fake_world(_worker, 2)
    expected = 72.0 / 40.0   # = 1.8
    for rank, r in enumerate(results):
        assert r is not None, f'rank {rank} returned None'
        assert abs(r - expected) < 1e-5, (
            f'rank {rank}: got {r}, expected {expected}.  '
            f'If you got 3.0, you computed mean-of-means (the bug this drills out).'
        )

    # Equal batches — weighted mean degenerates to plain mean.
    _equal_sums = [10.0, 20.0, 30.0]
    _equal_counts = [10, 10, 10]

    def _worker_equal(rank, world_size, dist_module, world):
        world.results[rank] = ex2_weighted_eval_mean(
            rank, world_size, dist_module, _equal_sums[rank], _equal_counts[rank])

    results_equal = _run_fake_world(_worker_equal, 3)
    expected_equal = (10 + 20 + 30) / 30   # = 2.0
    for rank, r in enumerate(results_equal):
        assert abs(r - expected_equal) < 1e-5, f'equal-batch rank {rank}: got {r}'

    # Zero-count rank — should not crash (assuming at least one rank has count > 0).
    _zero_sums = [50.0, 0.0, 100.0]
    _zero_counts = [10, 0, 20]   # rank 1 had no samples

    def _worker_zero(rank, world_size, dist_module, world):
        world.results[rank] = ex2_weighted_eval_mean(
            rank, world_size, dist_module, _zero_sums[rank], _zero_counts[rank])

    results_zero = _run_fake_world(_worker_zero, 3)
    expected_zero = (50 + 0 + 100) / (10 + 0 + 20)   # = 150/30 = 5.0
    for rank, r in enumerate(results_zero):
        assert abs(r - expected_zero) < 1e-5, f'zero-count case rank {rank}: got {r}'

    # Floating-point precision test — large counts.
    _large_sums = [1000.5, 2000.5, 3000.5, 4000.5]
    _large_counts = [100, 200, 300, 400]

    def _worker_large(rank, world_size, dist_module, world):
        world.results[rank] = ex2_weighted_eval_mean(
            rank, world_size, dist_module, _large_sums[rank], _large_counts[rank])

    results_large = _run_fake_world(_worker_large, 4)
    expected_large = sum(_large_sums) / sum(_large_counts)
    for rank, r in enumerate(results_large):
        assert abs(r - expected_large) < 1e-4, f'large-batch rank {rank}: got {r}, expected {expected_large}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_weighted_eval_mean(rank: int, world_size: int, dist_module,
                           local_loss_sum: float, local_count: int) -> float:
    stats = t.tensor([local_loss_sum, float(local_count)], dtype=t.float32)
    dist_module.all_reduce(stats, op=dist_module.ReduceOp.SUM)
    global_mean = stats[0] / stats[1]
    return global_mean.item()
```

**Why pack into one tensor.** Each `all_reduce` is a network round-trip; on a fast interconnect (NVLink, IB) each costs microseconds, but they add up across thousands of eval batches in a long training run. Packing two scalars into one tensor halves the all_reduce count.

**`local_loss_sum`, not `local_mean`.** This is the subtle bug fix vs ex1. If you reduce per-rank MEANS, the count weight is lost; the weighted-mean math no longer works. Pass the unreduced sum (or sum of loss × batch_size, depending on how you computed local loss).

**Float cast on count.** `all_reduce` requires a float tensor on gloo (and integer reduce is finicky on NCCL too). Cast `count` to `float32` on the way in; cast back if you really need an int (usually you don't — divide stays float).

**Division by zero edge case.** If EVERY rank has count 0 (rare), `stats[1] = 0` and division gives `inf`/`nan`. In practice the eval pipeline guards against this upstream; we trust the caller.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()